<a href="https://colab.research.google.com/github/Kalyani-asekar/NNDL/blob/main/practical10nndl_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
num_customers = 500
num_products = 100
num_samples = 5000

customer_ids = torch.randint(0, num_customers, (num_samples,)).to(device)
product_ids = torch.randint(0, num_products, (num_samples,)).to(device)
ratings = torch.randint(1, 6, (num_samples, 1)).float().to(device)

class NeuralCollaborativeFiltering(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=16):
        super(NeuralCollaborativeFiltering, self).__init__()
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.item_embed = nn.Embedding(num_items, embed_dim)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user, item):
        u_emb = self.user_embed(user)
        i_emb = self.item_embed(item)
        x = torch.cat([u_emb, i_emb], dim=-1)
        return self.fc(x)

model = NeuralCollaborativeFiltering(num_customers, num_products).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 30
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(customer_ids, product_ids)
    loss = criterion(outputs, ratings)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    sample_customer = torch.tensor([10]).to(device)
    sample_product = torch.tensor([45]).to(device)
    predicted_rating = model(sample_customer, sample_product)
    print("\nPredicted Rating for Customer 10 on Product 45:", predicted_rating.item())

Epoch [10/30], Loss: 3.8747
Epoch [20/30], Loss: 2.6488
Epoch [30/30], Loss: 2.2580

Predicted Rating for Customer 10 on Product 45: 2.6496007442474365
